[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# Relationships &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup: the catalog's models, the twelve books, and a database with
foreign keys switched on. Run it first, then the tasks in order.


In [1]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (CharField, ForeignKeyField, IntegerField, IntegrityError, ManyToManyField,
                    Model, OperationalError, SqliteDatabase)
from playhouse.test_utils import assert_query_count, count_queries

AUTHORS = [                                                         # name, the year of the first book
    ("Ursula Vance", 2014),
    ("Marco Pietra", 2009),
    ("Ines O'Brien", 1998),
    ("Kofi Mensah", 2015),
]

BOOKS = [                                                           # title, author, year, pages
    ("The Salt Road", "Ursula Vance", 2014, 312),
    ("Nightjar", "Ursula Vance", 2018, 244),
    ("The Quiet Engine", "Ursula Vance", 2021, 398),
    ("Stone and Tide", "Marco Pietra", 2009, 501),
    ("The Lantern Keeper", "Marco Pietra", 2016, 276),
    ("Riverwork", "Marco Pietra", 2022, 189),
    ("A Careful Fire", "Ines O'Brien", 1998, 420),
    ("The Long Field", "Ines O'Brien", 2004, 355),
    ("Winter Harbour", "Ines O'Brien", 2011, 263),
    ("The Drum Line", "Kofi Mensah", 2015, 198),
    ("Harmattan", "Kofi Mensah", 2019, 331),
    ("Small Machines", "Kofi Mensah", 2023, 287),
]

def sql(query):
    """The SQL a query will send, and the values that go with it, on one line."""
    statement, values = query.sql()
    return " ".join(statement.split()) + (f"  {values}" if values else "")

db = SqliteDatabase(":memory:", pragmas={"foreign_keys": 1})        # SQLite enforces nothing without this


class CatalogModel(Model):
    """Every model in the catalog names the database once, here."""

    class Meta:
        database = db


class Author(CatalogModel):
    name = CharField(max_length=60, unique=True)
    first_book = IntegerField()


class Book(CatalogModel):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="books")
    year = IntegerField(index=True)
    pages = IntegerField()

def build(database):
    """Create the tables and load the catalog, in one transaction."""
    database.create_tables([Author, Book])
    with database.atomic():
        Author.insert_many([{"name": name, "first_book": year} for name, year in AUTHORS]).execute()
        written = {author.name: author.id for author in Author.select()}
        Book.insert_many([{"title": title, "author": written[author], "year": year, "pages": pages}
                          for title, author, year, pages in BOOKS]).execute()


build(db)
print("peewee", peewee.__version__, "| foreign keys on:",
      db.execute_sql("PRAGMA foreign_keys").fetchone()[0] == 1,
      "|", Author.select().count(), "authors and", Book.select().count(), "books")


peewee 4.5.1 | foreign keys on: True | 4 authors and 12 books


**1.** Every book with its author, in one query.


In [2]:
with assert_query_count(1):
    listing = [f"{row.author.name}: {row.title}"
               for row in Book.select(Book, Author).join(Author).order_by(Author.name)]

for line in listing[:3]:
    print("  ", line)
print("  ...", len(listing), "books")


   Ines O'Brien: A Careful Fire
   Ines O'Brien: The Long Field
   Ines O'Brien: Winter Harbour
  ... 12 books


`select(Book, Author)` is what makes it one query. The `assert_query_count` passing is the proof,
and it would fail the moment somebody dropped the `Author` from the `select`.


**2.** The same loop, counted, with a plain join.


In [3]:
with count_queries() as counter:
    for row in Book.select().join(Author):
        row.author.name
print("plain join:", counter.count, "queries for", Book.select().count(), "books")
print("the SELECT list:", Book.select().join(Author).sql()[0].split(" FROM")[0])


plain join: 13 queries for 12 books
the SELECT list: SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages"


One query for the books, and one more for each book's author: twelve plus one. The `SELECT` list
names only `book` columns, so there was nothing on the row for `.author` to build an `Author` from.


**3.** An author's long books, through the backref.


In [4]:
marco = Author.get(Author.name == "Marco Pietra")
long_ones = marco.books.where(Book.pages > 300).order_by(Book.pages.desc())

print(sql(long_ones))
print([(written.title, written.pages) for written in long_ones])


SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages" FROM "book" AS "t1" WHERE (("t1"."author_id" = ?) AND ("t1"."pages" > ?)) ORDER BY "t1"."pages" DESC  [2, 300]
[('Stone and Tide', 501)]


The `backref` query already carries `author_id = ?`, and the `where` was added to it, so both
conditions are in one `WHERE` clause. `backref` gives a query, not a list, which is why it can be
filtered at all.


**4.** The same insert against two databases.


In [5]:
class Person(Model):
    name = CharField(max_length=60)

    class Meta:
        database = None                                             # bound below, twice


class Note(Model):
    text = CharField(max_length=80)
    person = ForeignKeyField(Person, backref="notes")

    class Meta:
        database = None


for database in (SqliteDatabase(":memory:"),
                 SqliteDatabase(":memory:", pragmas={"foreign_keys": 1})):
    database.bind([Person, Note])
    database.create_tables([Person, Note])
    setting = database.execute_sql("PRAGMA foreign_keys").fetchone()[0]
    try:
        Note.create(text="by nobody", person=777)
        print(f"  foreign_keys={setting} -> accepted, and there is no person 777")
    except IntegrityError as error:
        print(f"  foreign_keys={setting} -> peewee.IntegrityError: {error}")


  foreign_keys=0 -> accepted, and there is no person 777
  foreign_keys=1 -> peewee.IntegrityError: FOREIGN KEY constraint failed


One pair of models, two databases, one setting between them. `bind` is what let the same models be
used against both.


**5.** The flattened name, and a name of your own.


In [6]:
plain = Book.select(Book, Author).join(Author).objects().first()
print("flattened  -> title:", plain.title, "| name:", plain.name)

named = (Book.select(Book, Author.name.alias("author_name"))
             .join(Author).objects().first())
print("with alias -> title:", named.title, "| author_name:", named.author_name)


flattened  -> title: The Salt Road | name: Ursula Vance
with alias -> title: The Salt Road | author_name: Ursula Vance


`Book` has no `name` of its own, so the author's lands there unopposed. The `alias` is worth writing
anyway: it says what the column is, and it is the only thing that works once both models have a
column of that name.


**6.** A shelf, its topics, and the table between them.


In [7]:
class Subject(Model):
    name = CharField(max_length=40)

    class Meta:
        database = db


class Case(Model):
    name = CharField(max_length=40)
    subjects = ManyToManyField(Subject, backref="cases")

    class Meta:
        database = db


bridge = Case.subjects.get_through_model()
db.create_tables([Subject, Case, bridge])

shelf = Case.create(name="Shelf B2")
for name in ("Poetry", "Letters"):
    shelf.subjects.add(Subject.create(name=name))

print("subjects:", sorted(s.name for s in shelf.subjects))
print("rows in", bridge._meta.table_name + ":",
      [(row.case_id, row.subject_id) for row in bridge.select()])
print("cases for Poetry:", [c.name for c in Subject.get(Subject.name == "Poetry").cases])


subjects: ['Letters', 'Poetry']
rows in case_subject_through: [(1, 1), (1, 2)]
cases for Poetry: ['Shelf B2']


The through table holds pairs of keys and nothing else. Both directions read from the same rows,
which is what makes it a relationship rather than two lists that have to be kept in step.


---

&#8592; **Back to:** [Relationships](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/06-relationships.ipynb)  &nbsp;&middot;&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
